# 🌍 Asteroid Risk Analyzer — Exploratory Data Analysis

This notebook explores the NASA NeoWs dataset to understand patterns that distinguish potentially hazardous asteroids from safe ones.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.preprocessing import clean_and_prepare, get_feature_names

plt.style.use('dark_background')
sns.set_palette('husl')
%matplotlib inline

## 1. Load & Preview Data

In [ ]:
raw_df = load_data()
df = clean_and_prepare(raw_df)
feature_names = get_feature_names(df)

print(f'Shape: {df.shape}')
print(f'Features: {feature_names}')
df.head()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 2. Class Distribution

**Question:** How balanced is the dataset between hazardous and safe asteroids?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['is_potentially_hazardous_asteroid'].value_counts()
axes[0].bar(['Safe (0)', 'Hazardous (1)'], [counts.get(0, 0), counts.get(1, 0)],
            color=['#00ff7f', '#ff4444'])
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=12)

# Pie chart
axes[1].pie([counts.get(0, 0), counts.get(1, 0)],
            labels=['Safe', 'Hazardous'],
            colors=['#00ff7f', '#ff4444'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportions', fontsize=14)

plt.tight_layout()
plt.show()

print(f'Class imbalance ratio: {counts.get(0,0) / counts.get(1,1):.1f}:1 (safe:hazardous)')

## 3. Diameter vs Hazard

**Question:** Do hazardous asteroids tend to be larger?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot
safe_d = df[df['is_potentially_hazardous_asteroid'] == 0]['diameter_mean_km'].clip(0, 3)
haz_d = df[df['is_potentially_hazardous_asteroid'] == 1]['diameter_mean_km'].clip(0, 3)

axes[0].boxplot([safe_d, haz_d], labels=['Safe', 'Hazardous'],
                patch_artist=True,
                boxprops=dict(facecolor='#1a4a4a'),
                medianprops=dict(color='#00ffe7', linewidth=2))
axes[0].set_title('Diameter Distribution by Class (clipped at 3km)')
axes[0].set_ylabel('Mean Diameter (km)')

# Violin plot
axes[1].violinplot([safe_d, haz_d], positions=[0, 1], showmedians=True)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Safe', 'Hazardous'])
axes[1].set_title('Violin Plot: Diameter by Class')
axes[1].set_ylabel('Mean Diameter (km)')

plt.tight_layout()
plt.show()

print(f'Safe median diameter: {safe_d.median():.4f} km')
print(f'Hazardous median diameter: {haz_d.median():.4f} km')

## 4. Miss Distance Distribution

**Question:** How does miss distance separate the two classes?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

safe_m = df[df['is_potentially_hazardous_asteroid'] == 0]['miss_distance_km']
haz_m = df[df['is_potentially_hazardous_asteroid'] == 1]['miss_distance_km']

ax.hist(safe_m, bins=50, alpha=0.6, color='#00ff7f', label=f'Safe (n={len(safe_m):,})')
ax.hist(haz_m, bins=50, alpha=0.7, color='#ff4444', label=f'Hazardous (n={len(haz_m):,})')
ax.set_yscale('log')
ax.set_xlabel('Miss Distance (km)')
ax.set_ylabel('Count (log scale)')
ax.set_title('Miss Distance Histogram — Hazardous asteroids cluster at smaller distances')
ax.axvline(7_480_000, color='yellow', linestyle='--', label='PHA threshold (~7.48M km)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Velocity vs Miss Distance

**Question:** Do high-velocity, close-passing asteroids tend to be classified as hazardous?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

safe_data = df[df['is_potentially_hazardous_asteroid'] == 0]
haz_data = df[df['is_potentially_hazardous_asteroid'] == 1]

ax.scatter(safe_data['relative_velocity_km_s'],
           safe_data['miss_distance_km'] / 1e6,
           alpha=0.3, s=15, color='#00ff7f', label='Safe')
ax.scatter(haz_data['relative_velocity_km_s'],
           haz_data['miss_distance_km'] / 1e6,
           alpha=0.6, s=25, color='#ff4444', label='Hazardous')

ax.axhline(7.48, color='yellow', linestyle='--', alpha=0.5, label='PHA miss distance threshold')
ax.set_xlabel('Relative Velocity (km/s)')
ax.set_ylabel('Miss Distance (million km)')
ax.set_title('Velocity vs Miss Distance — Key Risk Dimensions')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

**Question:** Which features are most correlated with each other and with the target?

In [ ]:
num_cols = feature_names + ['is_potentially_hazardous_asteroid']
corr = df[[c for c in num_cols if c in df.columns]].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

print('\nCorrelation with target (is_potentially_hazardous_asteroid):')
print(corr['is_potentially_hazardous_asteroid'].sort_values(ascending=False))

## 7. Pairplot

**Question:** How do the most important features interact visually?

In [ ]:
top_features = ['diameter_mean_km', 'miss_distance_km', 'relative_velocity_km_s',
                'absolute_magnitude_h', 'is_potentially_hazardous_asteroid']
plot_df = df[[c for c in top_features if c in df.columns]].copy()
plot_df['is_potentially_hazardous_asteroid'] = plot_df['is_potentially_hazardous_asteroid'].map({0: 'Safe', 1: 'Hazardous'})

# Clip outliers for cleaner pairplot
plot_df['diameter_mean_km'] = plot_df['diameter_mean_km'].clip(0, 2)
plot_df['miss_distance_km'] = plot_df['miss_distance_km'].clip(0, 70_000_000)

g = sns.pairplot(plot_df, hue='is_potentially_hazardous_asteroid',
                 palette={'Safe': '#00ff7f', 'Hazardous': '#ff4444'},
                 plot_kws={'alpha': 0.4, 's': 15},
                 diag_kind='hist')
g.fig.suptitle('Pairplot — Top Features by Hazard Class', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Summary

**Key findings from EDA:**
1. Dataset is imbalanced (~18% hazardous) — requires class weighting in the model
2. Miss distance is the clearest separator between classes
3. Hazardous asteroids tend to be larger (higher diameter, lower magnitude)
4. Higher velocity doesn't alone predict hazard — miss distance is equally important
5. Diameter features (min, max, mean) are highly correlated — could use just one